# End-to-end fine-tuning on a free Colab GPU (RQ4)

Runtime → Change runtime type → **T4 GPU**. Assumes in Google Drive:
- `MyDrive/mmcu/multimodal-content-understanding.zip`: this repo (`zip -r ... . -x '.venv/*' 'data/*' 'features*/*' 'results*/*'`), or clone it from GitHub instead
- `MyDrive/mmcu/hateful_memes.zip`: the dataset (`img/` + `*.jsonl`)

Results land in `MyDrive/mmcu/runs/`. Copy them into `results/runs/` locally and re-run `scripts/analyze.py`; the fine-tuned models then appear next to the frozen-feature heads.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/mmcu'
!unzip -qo $D/multimodal-content-understanding.zip -d /content/mmcu
%cd /content/mmcu
!mkdir -p data && unzip -qo $D/hateful_memes.zip -d data/ && ls data/hateful_memes | head
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Sanity check: `--unfreeze 0` is the frozen setting trained end-to-end. It should land near the frozen `clip_interaction` result from your laptop.

In [ ]:
!python scripts/finetune.py --data data/hateful_memes --encoder clip-b32 --unfreeze 0 --epochs 8 --seed 0 --out $D/runs

How much does adapting the top transformer blocks help? (≈10–15 min per run on a T4 for B/32)

In [ ]:
for k in (2, 4):
    for seed in (0, 1, 2):
        !python scripts/finetune.py --data data/hateful_memes --encoder clip-b32 --unfreeze {k} --epochs 8 --seed {seed} --out $D/runs

Optional: a bigger backbone. ViT-B/16 sees 4× more image patches; ViT-L/14 needs `--batch-size 16 --grad-accum 2` on a T4.

In [ ]:
!python scripts/finetune.py --data data/hateful_memes --encoder clip-b16 --unfreeze 2 --epochs 8 --seed 0 --out $D/runs
!python scripts/latency.py --encoder clip-b32 --batch-sizes 1 32 --out $D/latency_t4_b32.json
!python scripts/latency.py --encoder clip-b16 --batch-sizes 1 32 --out $D/latency_t4_b16.json